# depends on DestinationStates

In [ ]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
)
import plotly.express as px


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
    STATS_CALCULATOR_REGISTRY,
    WETH,
    ETH_CHAIN,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks,
)


import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
)
from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_df,
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_highest_value_in_field_where,
    get_subset_not_already_in_column,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    get_state_by_one_block,
)
from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_active_destinations_by_autopool_by_block,
    fetch_pools_and_destinations_df,
)
from mainnet_launch.constants import (
    AutopoolConstants,
    ALL_AUTOPOOLS,
    AUTO_LRT,
    POINTS_HOOK,
    ChainData,
)


chain = ETH_CHAIN

possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    AutopoolDestinationStates,
    AutopoolDestinationStates.block,
    possible_blocks,
    where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
)

autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
full_destination_df = natural_left_right_using_where(
    DestinationTokens,
    Destinations,
    using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    where_clause=DestinationTokens.chain_id == chain.chain_id,
)

token_value_df = natural_left_right_using_where(
    DestinationTokenValues,
    TokenValues,
    using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)
token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)
token_value_df = pd.merge(
    token_value_df, token_df[["symbol", "decimals", "token_address"]], how="left", on="token_address"
)
token_value_df

2025-04-27 11:12:58,933 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:12:58,935 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-27 11:12:58,936 INFO sqlalchemy.engine.Engine [cached since 800.7s ago] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'date_trunc_2': 'day'}
2025-04-27 11:12:59,194 INFO sqlalchemy.engine.Engine COMMIT
2025-04-27 11:12:59,293 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:12:59,294 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopool_destination_states
            WHERE autopool_destination_states.chain_id = 1
        
2025-04-27 11:12:59,295 INFO sqlalchemy.engine.Engine [cached since 800.8s ago] {}
2025-04-27 11:12:59,483

In [ ]:
def build_autopool_balance_of_calls_by_destination(
    autopool_vault_address: str, destination_vault_addresses: list[str]
) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["balanceOf(address)(uint256)", autopool_vault_address],
            [((autopool_vault_address, destination_vault_address, "balanceOf"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


# DestinationVaultAddress.underlyingTotalSupply()
#  -> I'm 95% sure this is the total supply of the lp tokens, not the quantity of lp tokens staked for rewards
def build_destinations_underlyingTotalSupply_calls(destination_vault_addresses: list[str]) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["underlyingTotalSupply()(uint256)"],
            [((destination_vault_address, "underlyingTotalSupply"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


def fetch_destination_total_supply_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    all_active_destinations = set()

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]
        all_active_destinations.update(this_autopool_active_destinations)

    calls = build_destinations_underlyingTotalSupply_calls(list(all_active_destinations))
    destination_total_supply_df = get_raw_state_by_blocks(calls, missing_blocks, chain, include_block_number=True)
    return destination_total_supply_df


def fetch_autopool_balance_of_by_destination(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_balance_of_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_balance_of_calls.extend(
            build_autopool_balance_of_calls_by_destination(autopool_vault_address, this_autopool_active_destinations)
        )

    autopool_destination_balance_of_df = get_raw_state_by_blocks(
        autopool_balance_of_calls, missing_blocks, chain, include_block_number=True
    )
    return autopool_destination_balance_of_df


def _build_destination_points_calls(this_autopool_active_destinations: list[str], chain: ChainData) -> list[Call]:
    return [
        Call(
            POINTS_HOOK(chain),
            ["destinationBoosts(address)(uint256)", destination_vault_address],
            [((destination_vault_address, "points"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in this_autopool_active_destinations
    ]


def fetch_autopool_points_apr(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_points_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_points_calls.extend(_build_destination_points_calls(this_autopool_active_destinations, chain))

    autopool_points_df = get_raw_state_by_blocks(
        autopool_points_calls, missing_blocks, chain, include_block_number=True
    )
    return autopool_points_df


def _clean_summary_stats_info(success, summary_stats):
    if success is True:
        summary = {
            "destination": summary_stats[0],  # address
            "baseApr": summary_stats[1] / 1e18,
            "feeApr": summary_stats[2] / 1e18,
            "incentiveApr": summary_stats[3] / 1e18,
            "safeTotalSupply": summary_stats[4] / 1e18,
            "priceReturn": summary_stats[5] / 1e18,
            # "maxDiscount": summary_stats[6] / 1e18,
            # "maxPremium": summary_stats[7] / 1e18,
            # "ownedShares": summary_stats[8] / 1e18,
            "compositeReturn": summary_stats[9] / 1e18,  # in or out, ( not certain here)
            "pricePerShare": summary_stats[10] / 1e18,
            "pointsApr": None,  # set later
        }
        return summary
    else:
        return None


def _build_summary_stats_call(
    autopool: Autopools,
    dest: Destinations,
    direction: str = "out",
    amount: int = 0,
) -> Call:
    # /// @notice Gets the safe price of the underlying LP token
    # /// @dev Price validated to be inside our tolerance against spot price. Will revert if outside.
    # /// @return price Value of 1 unit of the underlying LP token in terms of the base asset
    # function getValidatedSafePrice() external returns (uint256 price);
    # getDestinationSummaryStats uses getValidatedSafePrice. So when prices are outside tolerance this function reverts

    # consider finding a version of this function that won't revert, (follow up, I am pretty sure that does not exist)
    if direction == "in":
        direction_enum = 0
    elif direction == "out":
        direction_enum = 1
    return_types = "(address,uint256,uint256,uint256,uint256,int256,int256,int256,uint256,int256,uint256)"

    # cleaning_function = build_summary_stats_cleaning_function(autopool)
    return Call(
        autopool.strategy_address,
        [
            f"getDestinationSummaryStats(address,uint8,uint256)({return_types})",
            dest.destination_vault_address,
            direction_enum,
            amount,
        ],
        [((autopool.autopool_vault_address, dest.destination_vault_address, direction), _clean_summary_stats_info)],
    )


def fetch_destination_summary_stats_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopools_orm: list[Autopools] = get_full_table_as_orm(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
    full_autopool_summary_stats_df = None

    for autopool in autopools_orm:
        all_summary_stats_calls = []
        this_autopool_destinations = autopool_to_all_ever_active_destinations[autopool.autopool_vault_address]
        for dest in this_autopool_destinations:
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "out"))
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "in"))

        autopool_summary_stats_df = get_raw_state_by_blocks(
            all_summary_stats_calls, missing_blocks, chain, include_block_number=True
        )

        if full_autopool_summary_stats_df is None:
            full_autopool_summary_stats_df = autopool_summary_stats_df.copy()
        else:
            full_autopool_summary_stats_df = pd.merge(
                full_autopool_summary_stats_df, autopool_summary_stats_df, on="block"
            )

    return full_autopool_summary_stats_df


def _fetch_autopool_and_destination_states(
    chain: ChainData,
    missing_blocks: list[int],
):
    autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)

    full_destination_df = natural_left_right_using_where(
        DestinationTokens,
        Destinations,
        using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
        where_clause=DestinationTokens.chain_id == chain.chain_id,
    )

    token_value_df = natural_left_right_using_where(
        DestinationTokenValues,
        TokenValues,
        using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
        where_clause=DestinationTokenValues.chain_id == chain.chain_id,
    )
    # not sure here on the way to specify only a subset of oclumsn? maybe with type hints? add later, eg columsn = none and
    token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)[
        ["symbol", "decimals", "token_address"]
    ]
    token_value_df = pd.merge(token_value_df, token_df, how="left", on="token_address")

    autopool_to_all_ever_active_destinations = fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks(
        chain, missing_blocks
    )

    destination_underlying_total_supply_df = fetch_destination_total_supply_df(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )
    autopool_destination_balance_of_df = fetch_autopool_balance_of_by_destination(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )  # only needed for autopool destination state, the rest can be extracted from the other tables

    autopool_points_df = fetch_autopool_points_apr(autopool_to_all_ever_active_destinations, missing_blocks, chain)

    autopool_summary_stats_df = fetch_destination_summary_stats_df(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )

    return (
        autopool_summary_stats_df,
        destination_underlying_total_supply_df,
        token_value_df,
        autopool_to_all_ever_active_destinations,
        autopool_points_df,
    )


(
    autopool_summary_stats_df,
    destination_underlying_total_supply_df,
    token_value_df,
    autopool_to_all_ever_active_destinations,
    autopool_points_df,
) = _fetch_autopool_and_destination_states(chain, missing_blocks)

2025-04-27 10:59:41,229 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 10:59:41,229 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-27 10:59:41,230 INFO sqlalchemy.engine.Engine [cached since 2.427s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x14ea54fa0>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-27 10:59:41,230 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-27 10:59:41,232 INFO

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [1]


2025-04-27 11:00:43,139 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:00:43,140 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-27 11:00:43,141 INFO sqlalchemy.engine.Engine [cached since 64.33s ago] {}
2025-04-27 11:00:43,324 INFO sqlalchemy.engine.Engine COMMIT


[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]

In [ ]:
all_new_destination_states = []
# autopool_summary_stats_df, destination_underlying_total_supply_df, token_value_df, autopool_to_all_ever_active_destinations
raw_destination_states_df = pd.merge(autopool_summary_stats_df, destination_underlying_total_supply_df, on="block")
raw_destination_states_df = pd.merge(raw_destination_states_df, autopool_points_df, on="block")


for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
    for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]:

        destination_tokens = full_destination_df[
            full_destination_df["destination_vault_address"] == dest.destination_vault_address
        ]["token_address"]

        just_this_destination_df = token_value_df[
            token_value_df["destination_vault_address"] == dest.destination_vault_address
        ].copy()

        # total value in the destination,
        just_this_destination_df["total_spot_value"] = (
            just_this_destination_df["quantity"] * just_this_destination_df["spot_price"]
        )
        just_this_destination_df["total_safe_value"] = (
            just_this_destination_df["quantity"] * just_this_destination_df["safe_price"]
        )
        just_this_destination_df["total_backing_value"] = (
            just_this_destination_df["quantity"] * just_this_destination_df["backing"]
        )

        this_destination_total_value = (
            just_this_destination_df.groupby("block")[["total_spot_value", "total_safe_value", "total_backing_value"]]
            .sum()
            .reset_index()
        )

        local_df = pd.merge(this_destination_total_value, raw_destination_states_df, on=["block"])

        def _extract_destination_states(row: pd.DataFrame) -> None:
            # try to pull the "in" and "out" stats, defaulting to an empty dict
            in_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "in"), {}) or {}
            out_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "out"), {}) or {}

            total_apr_in = in_summary_stats.get("compositeReturn")
            total_apr_out = out_summary_stats.get("compositeReturn")

            incentive_apr = in_summary_stats.get("incentiveApr")
            fee_apr = in_summary_stats.get("feeApr")
            base_apr = in_summary_stats.get("baseApr")

            price_per_share = in_summary_stats.get("pricePerShare")
            price_return = in_summary_stats.get("priceReturn")
            fee_plus_base_apr = None  # only for post autoUSD destinations
            safe_total_supply = in_summary_stats.get("safeTotalSupply")

            points_apr = row[(dest.destination_vault_address, "points")]
            underlying_total_supply = row[(dest.destination_vault_address, "underlyingTotalSupply")]

            # the value of a lp token
            underlying_safe_price = row["total_safe_value"] / underlying_total_supply
            underlying_spot_price = row["total_spot_value"] / underlying_total_supply
            underlying_backing = row["total_backing_value"] / underlying_total_supply

            safe_backing_discount = (underlying_safe_price - underlying_backing) / underlying_backing
            safe_spot_spread = (underlying_safe_price - underlying_spot_price) / underlying_safe_price
            spot_backing_discount = (underlying_spot_price - underlying_backing) / underlying_backing
            # return
            new_destination_state = DestinationStates(
                destination_vault_address=dest.destination_vault_address,
                block=row["block"],
                chain_id=chain.chain_id,
                incentive_apr=incentive_apr,
                fee_apr=fee_apr,
                base_apr=base_apr,
                points_apr=points_apr,
                fee_plus_base_apr=fee_plus_base_apr,
                total_apr_in=total_apr_in,
                total_apr_out=total_apr_out,
                underlying_token_total_supply=underlying_total_supply,
                safe_total_supply=safe_total_supply,
                price_per_share=price_per_share,
                price_return=price_return,
                underlying_safe_price=underlying_safe_price,
                underlying_spot_price=underlying_spot_price,
                underlying_backing=underlying_backing,
                safe_backing_discount=safe_backing_discount,
                spot_backing_discount=spot_backing_discount,
                safe_spot_spread=safe_spot_spread,
            )
            all_new_destination_states.append(new_destination_state)

        local_df.apply(_extract_destination_states, axis=1)


insert_avoid_conflicts(
    all_new_destination_states,
    DestinationStates,
    index_elements=[DestinationStates.block, DestinationStates.chain_id, DestinationStates.destination_vault_address],
)

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,6.621342e-06
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,13157.496413,12383.288934,1.036163,-0.000278,1.036163,1.036121,1.035875,0.000278,0.000237,4.066212e-05
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,13157.496413,12383.288934,1.036462,-0.000538,1.036462,1.036456,1.035906,0.000537,0.000531,6.483657e-06
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,13268.458946,12479.834854,1.036491,-0.000575,1.036491,1.036477,1.035896,0.000574,0.000560,1.429726e-05
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,13268.526874,12494.174709,1.036501,-0.000558,1.036501,1.036502,1.035924,0.000557,0.000558,-6.380341e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15003,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22327989,1,0.067254,0.002229,0.000000,0.0,None,0.062757,0.063927,6199.358769,5649.225061,1.000157,0.000000,1.000157,1.000204,1.001720,-0.001561,-0.001514,-4.723397e-05
15004,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22335153,1,0.069651,0.002030,0.000000,0.0,None,0.064716,0.066006,6029.808420,5463.737531,1.000052,0.000000,1.000052,1.000049,1.001777,-0.001722,-0.001725,2.761639e-06
15005,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22342307,1,0.069415,0.002526,0.000000,0.0,None,0.064999,0.066337,6079.772900,5501.052719,0.999989,0.000000,0.999989,1.000020,1.001777,-0.001784,-0.001753,-3.144923e-05
15006,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,22349480,1,0.069827,0.002311,0.000000,0.0,None,0.065155,0.066611,6062.134066,5496.063177,0.999858,0.000000,0.999858,0.999885,1.001804,-0.001943,-0.001915,-2.752155e-05


2025-04-27 11:12:23,552 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:12:23,890 INFO sqlalchemy.engine.Engine ROLLBACK


NotNullViolation: null value in column "incentive_apr" of relation "destination_states" violates not-null constraint
DETAIL:  Failing row contains (0xf3ae3c74EaD129e770A864CeE291A805b170bBe0, 21031733, 1, null, null, null, 0, null, null, null, 13142.64434205249, null, null, null, 1.036627562877249, 1.0366221304469656, 1.036718930273536, -8.813130890057634e-05, -9.337133117156794e-05, 5.240484121716512e-06).


In [ ]:
just_this_destination_df[["backing", "symbol"]]

In [ ]:
# maybe we get the apply map version?

In [ ]:
# class DestinationStates(Base):
#     __tablename__ = "destination_states"

#     destination_vault_address: Mapped[str] = mapped_column(primary_key=True)
#     block: Mapped[int] = mapped_column(primary_key=True)
#     chain_id: Mapped[int] = mapped_column(primary_key=True)
#     # information about the destination itself at this moment in time

#     incentive_apr: Mapped[float] = mapped_column(nullable=False)
#     fee_and_base_apr: Mapped[float] = mapped_column(nullable=False)
#     points_apr: Mapped[float] = mapped_column(nullable=True)

#     total_apr_in: Mapped[float] = mapped_column(
#         nullable=True
#     )  # get destination summaryStats (in, and out) are seperate calls
#     total_apr_out: Mapped[float] = mapped_column(nullable=True)

#     underlying_token_total_staked: Mapped[float] = mapped_column(nullable=True)
#     underlying_token_total_supply: Mapped[float] = mapped_column(nullable=False)
#     safe_total_supply: Mapped[float] = mapped_column(nullable=True)  # only for pre autoUSD destinations

#     # this is as lp tokens # via
#     underlying_safe_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_spot_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_backing: Mapped[float] = mapped_column(nullable=False)
#     denominated_in: Mapped[str] = mapped_column(nullable=False) # should live in the destination

#     safe_backing_discount: Mapped[float] = mapped_column(nullable=True)
#     safe_spot_spread: Mapped[float] = mapped_column(nullable=True)
#     spot_backing_discount: Mapped[float] = mapped_column(nullable=True)

#     __table_args__ = (
#         ForeignKeyConstraint(["block", "chain_id"], ["blocks.block", "blocks.chain_id"]),
#         ForeignKeyConstraint(
#             ["destination_vault_address", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#     )

In [ ]:
# drop this
AutopoolDestinationStates_new_partial_rows = []

def _to_apply_over_each_row(row:dict):
    
    for autopool_dest_tuple, balance_of in row.items():
        if autopool_dest_tuple != 'block':
            autopool_vault_address, destination_vault_address = col # unpack the tuple

            AutopoolDestinationStates_new_partial_rows.append(
                        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':destination_vault_address, 
            'block': row['block'],
            'amount': balance_of,
            # this will raise an error if trying to insert into AutopoolDestinationStates because nullable = false 
            'total_safe_value': None,
            'total_spot_value': None,
            'total_backing_value': None,
            'percent_ownership': -100.0, # depends on destination_states.underlying_token_total_supply
        }

            )

# t



for col in autopool_destination_balance_of_df.columns:
    this_autopool_destination_balance_of = []
    autopool_balance_columns = autopool_destination_balance_of_df[col]
    autopool_vault_address, destination_vault_address = col # unpack
    this_autopool_destination_balance_of.append(
        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':

        }

    )



In [ ]:
autopool_to_all_ever_active_destinations.keys()

In [ ]:
autopool_to_all_ever_active_destinations.keys

In [ ]:
# def build_pool_token_spot_price_calls(
#     chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
# ) -> list[Call]:

#     return [
#         Call(
#             ROOT_PRICE_ORACLE(chain),
#             ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
#             [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
#         )
#         for (pool_address, token_address) in zip(pool_addresses, token_addresses)
#     ]


# spot_price_calls = build_pool_token_spot_price_calls(
#     chain, full_destination_df["pool"], full_destination_df["token_address"]
# )
# destination_token_spot_price_df = get_raw_state_by_blocks(
#     spot_price_calls, possible_blocks, chain, include_block_number=True
# )
# destination_token_spot_price_df


# def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
#     return [
#         Call(
#             dest,
#             "underlyingReserves()(address[],uint256[])",
#             [
#                 (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
#                 (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
#             ],
#         )
#         for dest in destinations
#     ]


# underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
# underlying_reserves_df = get_raw_state_by_blocks(
#     underlying_reserves_calls, possible_blocks, chain, include_block_number=True
# )
# underlying_reserves_df